# Lesson 03 - Ajan Tasarım Kalıpları

Bu derste, etkili yapay zeka ajanları oluşturmak için üç temel tasarım kalıbını inceliyoruz:

1. **Net Ajan Talimatları** — Ajan davranışını yönlendiren kesin, rol tanımlayıcı istemler oluşturmak  
2. **Pydantic Modelleriyle Yapılandırılmış Çıktı** — Ajanların öngörülebilir, doğrulanmış veri dönmesini sağlamak  
3. **Tek Sorumluluk Ajanları** — Her biri bir işi iyi yapan odaklı ajanlar tasarlamak

Her kalıbı, bir **seyahat destinasyonu önericisi** senaryosuna uygulayacağız ve kademeli olarak destinasyon öneren, uygunluğu kontrol eden ve lojistiği yöneten bir sistem inşa edeceğiz.


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Kalıp 1: Net Ajan Talimatları

En etkili kalıp aynı zamanda en basit olandır: ajanın için net, detaylı talimatlar yazmak.

İyi talimatlar şunları tanımlar:
- **Kim** olduğu (kişilik ve ton)
- **Ne** yapması gerektiği (adım adım sorumluluklar)
- **Nasıl** davranması gerektiği (kısıtlamalar ve stil)

Aşağıda, ürettiği her yanıtı şekillendiren açık talimatlarla bir seyahat konsiyerj ajanı oluşturuyoruz.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Desen 2: Pydantic Modelleri ile Yapılandırılmış Çıktı

Serbest biçimli metin sohbet için faydalıdır, ancak alt sistemler yapılandırılmış verilere ihtiyaç duyar.
**Pydantic modellerini** bir **araç fonksiyonu** ile eşleştirerek:

- Ajanın çıktısı için kesin bir şema tanımlayabiliriz
- Yanıtları otomatik olarak doğrulayabiliriz
- Ajan sonuçlarını uygulama mantığına güvenilir şekilde entegre edebiliriz

Ayrıca, ajanın önerilerini gerçek verilere dayandırması için bir varış yeri ayrıntılarını döndüren bir araç da tanıtıyoruz.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Desen 3: Tek Sorumluluklu Ajanlar

Karmaşık görevler, her biri tek bir sorumluluğa sahip birden çok odaklanmış ajanın iş bölümü yapmasından fayda sağlar:

- Yerler ve uygunluk hakkında bilgi sahibi bir **Varış Uzmanı**
- Uçuşlar, oteller ve seyahat programlarını yöneten bir **Lojistik Planlayıcı**

Bu, yazılım mühendisliği prensibi olan *sorumluluk ayrımı* prensibini yansıtır — her bir ajan bağımsız olarak test edilmesi, bakımı ve geliştirilmesi daha kolaydır.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Özet

Bu derste üç ajan tasarım desenini bir seyahat öneri senaryosuna uyguladık:

| Desen | Temel Fikir | Fayda |
|---|---|---|
| **Açık Talimatlar** | Kişilik, sorumluluklar ve kısıtlamaları baştan tanımla | Tutarlı, markaya uygun ajan davranışı |
| **Yapılandırılmış Çıktı** | Yanıt formatı olarak Pydantic modellerini kullan | Doğrulanmış, makine tarafından okunabilir sonuçlar |
| **Tek Sorumluluk** | Her ajana odaklanmış tek bir görev ver | Test etmeyi, bakımını ve bileştirmeyi kolaylaştırır |

Bu desenler doğal olarak birleşir — açık talimatları yapılandırılmış çıktı ile tek sorumluluklu bir ajanın içinde birleştirerek sağlam, üretime hazır sistemler oluşturabilirsiniz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba gösterilse de, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayınız. Orijinal belge, kendi ana dilinde yetkili kaynak olarak kabul edilmelidir. Önemli bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanılması sonucu oluşabilecek her türlü yanlış anlama veya yorumdan dolayı sorumluluk kabul edilmez.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
